In [21]:
import numpy as np 
import torch.nn.functional as F
import torch

Q = np.random.rand(10, 128)
K = np.random.rand(10, 128)
V = np.random.rand(10, 128)

def softmax(x):
    max_vals = np.max(x, axis=-1, keepdims=True)
    top = np.exp(x - max_vals) 
    bottom = np.sum(np.exp(x - max_vals), keepdims=True, axis=-1)
    return top / bottom

In [23]:
def softmax_causal(x):
    m_dim, n_dim = x.shape 
    assert len(x.shape) == 2 and "only work for dim2 for now"
    result = np.zeros(x.shape)
    for m in range(m_dim):
        max_val = np.max(x[m, 0:(m+1)])
        sum_val = np.sum(np.exp(x[m, 0:(m+1)] - np.array([max_val])))
        for n in range(n_dim):
            if n <= m: 
                result[m, n] = np.exp(x[m, n] - max_val) / sum_val 
            else: 
                result[m, n] = 0
    return result

def my_attention(Q, K, V): 
    dim_sequence, dim_model = Q.shape 
    _, dim_embedding = V.shape 
    output = np.zeros((dim_sequence, dim_embedding))
    # these come from the linalg.generic loop (i, j is inferred)
    for i in range(dim_sequence):
        for j in range(dim_embedding):
            # here emit the first linalg.matmul for the first attention
            pre_softmax = Q @ K.T
            # do the softmax calculation here, it it possible that you have emit to linalg, one to find max, and the other one to calculate the causal softmax
            softmax_calculation = softmax_causal(pre_softmax)
            # emit this loop through the linalg.generic, within the inner linalg.generic
            for k in range(dim_sequence):
                output[i, j] += softmax_calculation[i, k] * V[k, j]
            
    return output

my_atten = my_attention(Q, K, V)
torch_atten = F.scaled_dot_product_attention(torch.from_numpy(Q), torch.from_numpy(K), torch.from_numpy(V), is_causal=True, scale=1)
print(f"My Implementation:\n {my_atten}")
print(f"Pytorch Implementation:\n {torch_atten}")
print(f"All close: {torch.allclose(torch.from_numpy(my_atten), torch_atten)}")

My Implementation:
 [[0.85541211 0.77541741 0.93996509 ... 0.8766301  0.09325708 0.28927122]
 [0.8695674  0.7212076  0.87331163 ... 0.8356228  0.14216625 0.26342806]
 [0.57259985 0.44122285 0.45515028 ... 0.57073528 0.63654701 0.41274222]
 ...
 [0.52009702 0.44638406 0.40319804 ... 0.55191094 0.73997489 0.55430074]
 [0.50703729 0.40947185 0.54161755 ... 0.65433109 0.59010933 0.41838467]
 [0.56692116 0.2056075  0.63590044 ... 0.67983003 0.45442581 0.3840262 ]]
Pytorch Implementation:
 tensor([[0.8554, 0.7754, 0.9400,  ..., 0.8766, 0.0933, 0.2893],
        [0.8696, 0.7212, 0.8733,  ..., 0.8356, 0.1422, 0.2634],
        [0.5726, 0.4412, 0.4552,  ..., 0.5707, 0.6365, 0.4127],
        ...,
        [0.5201, 0.4464, 0.4032,  ..., 0.5519, 0.7400, 0.5543],
        [0.5070, 0.4095, 0.5416,  ..., 0.6543, 0.5901, 0.4184],
        [0.5669, 0.2056, 0.6359,  ..., 0.6798, 0.4544, 0.3840]],
       dtype=torch.float64)
All close: True


In [76]:
my_atten.shape

(10, 128)